# Data Loaders

This notebook handles format-specific data loading and standardization for different trajectory data sources.

## Supported Formats:
- **GPX** (OpenStreetMap traces)
- **Geolife CSV** (Walking/running trajectories from Microsoft Geolife dataset)

## Output:
Standardized CSV files with consistent schema:
- `latitude`, `longitude`, `altitude`, `timestamp`
- `user_id`, `source_file`, `source_format`, `trajectory_id`

In [ ]:
import os
import warnings

import gpxpy
import pandas as pd

# Import username extraction module for OSM trace user identification
from helpers.gpx_username_extractor import GPXUsernameExtractor

warnings.filterwarnings("ignore")

In [ ]:
# Select which data sources to load
# Comment out sources you don't want to process
# Defaults reproduce the OSM Rome run. The environment overrides below let a
# second corpus be ingested without editing this cell and without disturbing
# the first -- each corpus keeps its own files/csv-<name> directory.
#   TRAJECTO_SOURCES=geolife
#   TRAJECTO_GEOLIFE_CSV=../data/geolife/.../walking_trajectories_candidates.csv
LOAD_SOURCES = os.environ.get('TRAJECTO_SOURCES', 'gpx').split(',')

# Configuration
CONFIG = {
    'input_dir': '../data/osm-rome/rome_center',  # GPX input directory (combines all .gpx files into combined.gpx)
    'output_dir': '../files/raw',  # Standardized raw data output
    'geolife_csv': os.environ.get(
        'TRAJECTO_GEOLIFE_CSV',
        '../data/geolife/Geolife Trajectories 1.3/walking_trajectories_1-20.csv'),
}

# Create output directory
os.makedirs(CONFIG['output_dir'], exist_ok=True)
print(f"Output directory: {CONFIG['output_dir']}")
print(f"Loading sources: {LOAD_SOURCES}")


## 1. GPX Data Loader

Handles GPX files from OpenStreetMap. Features:
- Combines multiple GPX files into one
- Extracts usernames from OSM URL tags
- Converts to standardized format

In [ ]:
def combine_gpx_files(input_dir):
    """
    Combine all GPX files in directory into a single combined.gpx file.
    Returns the path to the combined file.
    """
    gpx_files = sorted([f for f in os.listdir(input_dir) if f.endswith('.gpx') and f != 'combined.gpx'])

    if len(gpx_files) == 0:
        # Check if combined.gpx already exists
        combined_path = os.path.join(input_dir, 'combined.gpx')
        if os.path.exists(combined_path):
            print("Using existing combined.gpx")
            return combined_path
        print(f"No GPX files found in {input_dir}")
        return None

    if len(gpx_files) == 1:
        print(f"Single GPX file found: {gpx_files[0]}")
        return os.path.join(input_dir, gpx_files[0])

    print(f"Combining {len(gpx_files)} GPX files...")

    # Parse first file as base
    first_file = os.path.join(input_dir, gpx_files[0])
    with open(first_file, encoding='utf-8') as f:
        combined_gpx = gpxpy.parse(f)

    print(f"  Base: {gpx_files[0]} ({len(combined_gpx.tracks)} tracks)")

    # Add tracks from remaining files
    for gpx_file in gpx_files[1:]:
        file_path = os.path.join(input_dir, gpx_file)
        with open(file_path, encoding='utf-8') as f:
            gpx = gpxpy.parse(f)
        for track in gpx.tracks:
            combined_gpx.tracks.append(track)
        print(f"  Added {len(gpx.tracks)} tracks from: {gpx_file}")

    # Write combined file
    combined_path = os.path.join(input_dir, 'combined.gpx')
    with open(combined_path, 'w', encoding='utf-8') as f:
        f.write(combined_gpx.to_xml())

    # Delete original files
    for gpx_file in gpx_files:
        os.remove(os.path.join(input_dir, gpx_file))
        print(f"  Deleted: {gpx_file}")

    print(f"Created combined.gpx with {len(combined_gpx.tracks)} total tracks")
    return combined_path

In [ ]:
def load_gpx_to_dataframe(gpx_path):
    """
    Load GPX file and convert to standardized DataFrame.
    Each track is loaded as a single trajectory (all segments combined).
    Notebook 1's split_on_significant_gaps() handles meaningful splits.
    """
    # Extract usernames from tracks
    username_extractor = GPXUsernameExtractor()
    tracks_metadata = username_extractor.parse_gpx_tracks_with_users(gpx_path)

    print(f"\nFound {len(tracks_metadata)} tracks in GPX file")
    print(username_extractor.generate_summary_report(tracks_metadata))

    # Parse GPX file
    with open(gpx_path, encoding='utf-8') as f:
        gpx = gpxpy.parse(f)

    all_points = []

    for track_meta in tracks_metadata:
        track_idx = track_meta['track_index']
        username = track_meta['username']
        track = gpx.tracks[track_idx]

        # Use track-level trajectory ID (not segment-level) to avoid over-fragmentation
        # Notebook 1's split_on_significant_gaps() will handle meaningful splits
        for segment in track.segments:
            for point in segment.points:
                all_points.append({
                    'latitude': point.latitude,
                    'longitude': point.longitude,
                    'altitude': point.elevation if point.elevation else 0,
                    'timestamp': point.time,
                    'user_id': username,
                    'source_file': os.path.basename(gpx_path),
                    'source_format': 'gpx',
                    'trajectory_id': f"gpx_{track_idx}"
                })

    df = pd.DataFrame(all_points)
    print(f"\nLoaded {len(df)} points from GPX")
    return df

In [ ]:
# Process GPX files
gpx_df = None

if 'gpx' in LOAD_SOURCES:
    if os.path.exists(CONFIG['input_dir']):
        gpx_path = combine_gpx_files(CONFIG['input_dir'])
        if gpx_path:
            gpx_df = load_gpx_to_dataframe(gpx_path)
            print("\nGPX Data Summary:")
            print(f"  Total points: {len(gpx_df)}")
            print(f"  Unique users: {gpx_df['user_id'].nunique()}")
            print(f"  Unique trajectories: {gpx_df['trajectory_id'].nunique()}")
    else:
        print(f"GPX input directory not found: {CONFIG['input_dir']}")
else:
    print("Skipping GPX loading (not in LOAD_SOURCES)")

## 2. Geolife CSV Loader

Handles Geolife dataset walking/running trajectories.
Input format: `user_id, trajectory_file, latitude, longitude, altitude, datetime, mode`

In [ ]:
def load_geolife_csv(csv_path):
    """
    Load Geolife walking trajectories CSV and convert to standardized format.
    """
    if not os.path.exists(csv_path):
        print(f"Geolife CSV not found: {csv_path}")
        return None

    # Load CSV
    df = pd.read_csv(csv_path)
    print(f"Loaded {len(df)} points from Geolife CSV")
    df[['date', 'time']] = df['datetime'].str.split('T', expand=True)

    # Standardize column names
    standardized = pd.DataFrame({
        'latitude': df['latitude'],
        'longitude': df['longitude'],
        'altitude': df['altitude'],
        'timestamp': df['datetime'],
        'date': df['date'],
        'time': df['time'],
        'user_id': df['user_id'].astype(str),
        'source_file': df['trajectory_file'],
        'source_format': 'geolife',
        'trajectory_id': df['user_id'].astype(str) + '_' + df['trajectory_file'].str.replace('.plt', '')
    })

    # Add mode as metadata (walk/run)
    standardized['mode'] = df['mode']

    return standardized

In [ ]:
# Process Geolife CSV
geolife_df = None

if 'geolife' in LOAD_SOURCES:
    geolife_df = load_geolife_csv(CONFIG['geolife_csv'])

    if geolife_df is not None:
        print("\nGeolife Data Summary:")
        print(f"  Total points: {len(geolife_df)}")
        print(f"  Unique users: {geolife_df['user_id'].nunique()}")
        print(f"  Unique trajectories: {geolife_df['trajectory_id'].nunique()}")
        if 'mode' in geolife_df.columns:
            print(f"  Modes: {geolife_df['mode'].value_counts().to_dict()}")
else:
    print("Skipping Geolife loading (not in LOAD_SOURCES)")

## 3. Export Standardized Data

Save loaded data in standardized format for the preprocessing pipeline.

In [ ]:
# Export GPX data
if gpx_df is not None and len(gpx_df) > 0:
    gpx_output = os.path.join(CONFIG['output_dir'], 'gpx_raw.csv')
    gpx_df.to_csv(gpx_output, index=False)
    print(f"Exported GPX data to: {gpx_output}")

# Export Geolife data
if geolife_df is not None and len(geolife_df) > 0:
    geolife_output = os.path.join(CONFIG['output_dir'], 'geolife_raw.csv')
    geolife_df.to_csv(geolife_output, index=False)
    print(f"Exported Geolife data to: {geolife_output}")

In [ ]:
# Final summary
print("\n" + "="*70)
print("DATA LOADING COMPLETE")
print("="*70)

if gpx_df is not None:
    print(f"\nGPX: {len(gpx_df)} points, {gpx_df['user_id'].nunique()} users")

if geolife_df is not None:
    print(f"Geolife: {len(geolife_df)} points, {geolife_df['user_id'].nunique()} users")

print(f"\nOutput directory: {CONFIG['output_dir']}")
print("\nNext step: Run 1-trajectory-preprocessing.ipynb")